# Sprint 1 — Tester Albert sur les 20 articles de référence

Ce carnet fait trois choses :
1. il affiche **vos quotas Albert** et **la liste des modèles disponibles** ;
2. il fait classer les 20 articles de référence (`tests/reference_20.csv`) par un modèle d'Albert, avec les consignes de `config/consignes/` ;
3. il compare les étiquettes du modèle aux vôtres, champ par champ.

**Avant de commencer (une seule fois)** : cliquez sur l'icône de clé 🔑 dans la barre de gauche de Colab, ajoutez un secret nommé `ALBERT_API_KEY`, collez-y votre clé Albert, et activez « Accès au notebook ». La clé ne s'affiche jamais dans le carnet.

Puis : menu **Exécution > Tout exécuter**. Comptez 5 à 10 minutes.

In [ ]:
!pip install -q openai trafilatura pandas

In [ ]:
import io, json, re, time
import pandas as pd
import requests
from openai import OpenAI
from google.colab import userdata

BRANCHE = "v2-sprint1"   # mettre "main" une fois la branche fusionnee
DEPOT = f"https://raw.githubusercontent.com/GabrielPlassat/tableaubord_innov/{BRANCHE}"
ADRESSE_ALBERT = "https://albert.api.etalab.gouv.fr/v1"

CLE = userdata.get("ALBERT_API_KEY")
client = OpenAI(base_url=ADRESSE_ALBERT, api_key=CLE)
print("Connexion prête.")

## 1. Vos quotas et les modèles disponibles

In [ ]:
r = requests.get(f"{ADRESSE_ALBERT}/me/info", headers={"Authorization": f"Bearer {CLE}"}, timeout=30)
print("Statut :", r.status_code)
info = r.json() if r.ok else {}
print(json.dumps(info.get("limits", info), indent=2, ensure_ascii=False)[:3000])

In [ ]:
modeles = []
for m in client.models.list().data:
    d = m.model_dump()
    modeles.append({"id": d.get("id"), "type": d.get("type", ""), "aliases": ", ".join(d.get("aliases") or [])})
modeles = pd.DataFrame(modeles)
display(modeles)

## 2. Choisir le modèle à tester

Par défaut, le carnet prend le premier modèle de type `text-generation`. Pour en tester un autre, écrivez son `id` entre les guillemets ci-dessous, puis relancez à partir de cette cellule. Testez-en 2 ou 3 (un petit, un moyen, un grand) pour comparer.

In [ ]:
MODELE = ""   # par exemple "openweight-medium" ; vide = choix automatique
if not MODELE:
    candidats = modeles[modeles["type"].str.contains("text-generation", na=False)]["id"].tolist()
    MODELE = candidats[0] if candidats else modeles["id"].iloc[0]
print("Modèle testé :", MODELE)

## 3. Classer les 20 articles

In [ ]:
reference = pd.read_csv(f"{DEPOT}/tests/reference_20.csv", dtype=str).fillna("")
passe1 = requests.get(f"{DEPOT}/config/consignes/passe1_v1.txt").text
passe2 = requests.get(f"{DEPOT}/config/consignes/passe2_v1.txt").text
techno = pd.read_csv(f"{DEPOT}/config/technologies.csv")
liste_techno = ", ".join(techno["nom"])
print(len(reference), "articles de référence chargés.")

import trafilatura

def lire_article(url):
    """Renvoie (titre, texte). Le texte n'est gardé qu'en mémoire, le temps du test."""
    try:
        brut = trafilatura.fetch_url(url)
        if not brut:
            return "", ""
        texte = trafilatura.extract(brut) or ""
        meta = trafilatura.extract_metadata(brut)
        return (meta.title if meta and meta.title else ""), texte
    except Exception as e:
        return "", ""

def demander(consigne, max_tokens=1500):
    rep = client.chat.completions.create(model=MODELE, temperature=0,
        max_tokens=max_tokens, messages=[{"role": "user", "content": consigne}])
    return rep.choices[0].message.content or ""

def extraire_json(texte):
    debut, fin = texte.find("{"), texte.rfind("}")
    if debut < 0 or fin < 0:
        return None
    try:
        return json.loads(texte[debut:fin + 1])
    except json.JSONDecodeError:
        return None

In [ ]:
resultats = []
for _, art in reference.iterrows():
    titre, texte = lire_article(art["url"])
    titre = titre or art["titre_court"]
    ligne = {"num": art["num"], "titre": art["titre_court"], "texte_lu": len(texte)}
    try:
        t0 = time.time()
        r1 = demander(passe1.replace("<<TITRE>>", titre).replace("<<EXTRAIT>>", texte[:500]), 30)
        ligne["passe1"] = r1.strip().split("|")[0].strip().upper()
        r2 = demander(passe2.replace("<<TECHNOLOGIES>>", liste_techno).replace("<<TITRE>>", titre)
                      .replace("<<LANGUE>>", art["langue"]).replace("<<TEXTE>>", texte[:6000]))
        fiche = extraire_json(r2)
        ligne["json_valide"] = fiche is not None
        ligne["duree_s"] = round(time.time() - t0, 1)
        if fiche:
            axes = fiche.get("axes") or []
            ligne.update({
                "lentille_principale": fiche.get("lentille_principale") or "",
                "lentille_secondaire": fiche.get("lentille_secondaire") or "AUCUNE",
                "iceberg": fiche.get("iceberg") or "",
                "aire_sujet": ";".join(fiche.get("aire_sujet") or []),
                "nature": fiche.get("nature") or "",
                "axe": axes[0].get("code", "") if axes else "AUCUN",
                "pole": axes[0].get("pole", "") if axes else "",
                "precipitation": fiche.get("precipitation") or "",
                "rebond": (fiche.get("rebond_type") or "") if fiche.get("rebond") == "MENTIONNE" else "NON_MENTIONNE",
                "resume": fiche.get("resume", ""),
            })
    except Exception as e:
        ligne["erreur"] = str(e)[:200]
    resultats.append(ligne)
    print(art["num"], "·", art["titre_court"], "·", ligne.get("passe1"), "· JSON", ligne.get("json_valide"))
    time.sleep(2)   # ménage les quotas
modele_df = pd.DataFrame(resultats)

## 4. Comparer aux étiquettes de référence

In [ ]:
champs = ["lentille_principale", "lentille_secondaire", "iceberg", "aire_sujet", "nature",
          "axe", "pole", "precipitation", "rebond"]
comp = reference.merge(modele_df, on="num", suffixes=("_ref", "_mod"))
scores = {}

# Passe 1 : un OUI de référence doit être OUI ou DOUTE ; un NON doit être NON
p1 = comp.apply(lambda r: (r["passe1"] in ("OUI", "DOUTE")) if r["pertinent"] == "OUI"
                else (r["passe1"] == "NON"), axis=1)
scores["passe 1 (tri)"] = p1.mean()
pertinents = comp[comp["pertinent"] == "OUI"]
scores["JSON valide"] = pertinents["json_valide"].fillna(False).mean()

for c in champs:
    ref, mod = pertinents[f"{c}_ref"] if f"{c}_ref" in pertinents else pertinents[c], pertinents.get(f"{c}_mod")
    if mod is None:
        continue
    if c == "aire_sujet":
        ok = [r in str(m).split(";") for r, m in zip(ref, mod)]
    else:
        ok = [str(r) == str(m) for r, m in zip(ref, mod)]
    scores[c] = sum(ok) / len(ok)

tableau = pd.DataFrame({"accord": scores}).style.format("{:.0%}")
print("Modèle :", MODELE)
display(tableau)
print("Articles dont le texte n'a pas pu être lu :", list(modele_df.loc[modele_df["texte_lu"] == 0, "num"]))

In [ ]:
# Détail article par article, et fichier à télécharger
detail_cols = ["num", "titre"] + [x for c in champs for x in (f"{c}_ref", f"{c}_mod") if x in comp.columns]
display(comp[detail_cols])
nom = f"resultats_{MODELE.replace('/', '_')}.csv"
comp.to_csv(nom, index=False)
from google.colab import files
files.download(nom)

## Comment lire le résultat

- Les seuils visés sont ceux de L2, section 6.2 : environ 80 % sur la lentille principale et la nature, 65 % sur l'iceberg, 70 % sur les axes et la précipitation.
- Sur 20 articles, un écart de 5 % = un seul article : c'est un premier coup de sonde, pas une évaluation. L'évaluation complète se fera sur 100 articles au Sprint 3.
- **Les sites qui bloquent la lecture automatique** (texte lu = 0) faussent le test : le modèle ne voit que le titre. Notez-les.
- Envoyez le tableau des accords à Claude : il suffit de le copier-coller.